# 2026-09-02_2 线1：U1 连续锚定 + 时期匹配显著性 fate–pathway 连接

目的：Fig3 不给 pathway 分命运；pathway 只作功能空间，比较 Fig2 三组 DEG 的功能特有/共享。

**连续锚定窗口**（方向由 leading edge 机制自动保证 NES>0；基因可换人 union）：
- **DM / DAL**：onset(发育期首个锚定 stage) → **Young**，每个可检验 stage 的 leading edge 都含 ≥1 个该命运 DEG；
  （DM 与 DAL 在 pathway 层规则完全相同，命运纯由锚定基因决定；不再区分 DM-divergent）
- **AAM**：Young 和 Old 都锚定；
- not_testable 不打断连续性；tested 但未锚定才打断。

**U1 时期匹配显著性过滤**（此版正式采纳）：
- onset 在胚胎期（DM/DAL）：胚胎期 ≥1 显著锚定 stage **且** 成年期(Young/Old) ≥1 显著锚定 stage；
- onset 在成年期（AAM）：成年期 ≥1 显著锚定 stage。
- 显著=padj<0.05 且方向一致；因 V1 连续锚定下窗口内每个 tested stage 都锚定，『窗口内显著』即『显著且锚定』。

措辞：U1 有显著性支撑，可用 enrichment；解释单位=module。全部标注列保留（改档零重跑）。

In [ ]:
suppressPackageStartupMessages({library(dplyr); library(tidyr); library(readr); library(purrr)})

in_dir  <- "/project/work/2026-09-01_GSEA_test/Data/"
out_dir <- in_dir
gsea_file <- "/project/work/2026-07-29_trajectory_class/Data/2026-08-02_1_lifecycle_directional_GSEA_all_results.csv"
dev_stages <- c("Early","Mid","Late")
adult_stages <- c("Young","Old")
avail <- list(AC=c("Early","Mid","Late","Young","Old"), Cone=c("Early","Mid","Late","Young","Old"),
              HC=c("Early","Mid","Late","Young","Old"), Rod=c("Mid","Late","Young","Old"),
              MG=c("Mid","Late","Young","Old"), BC=c("Late","Young","Old"))

detail <- read_csv(file.path(in_dir, "2026-09-02_1_anchor_detail_long.csv"), show_col_types = FALSE)
cat("明细表行数:", nrow(detail), "\n")

In [ ]:
# ---- 每个 pathway×celltype 的可检验 stage + 五期 NES ------------------------
gsea <- read_csv(gsea_file, show_col_types = FALSE)
tested_tbl <- gsea %>%
  filter(gsea_status %in% c("significant_Central_macular","significant_Peripheral","tested_not_significant")) %>%
  group_by(celltype, database, pathway) %>%
  summarise(tested_stages = list(stage), .groups="drop")
nes_wide <- gsea %>% filter(stage %in% unlist(avail)) %>%
  select(celltype, database, pathway, stage, NES) %>%
  pivot_wider(names_from=stage, values_from=NES, names_prefix="NES_")

In [ ]:
# ---- 连接聚合：每个 celltype×direction×pathway×subclass 一行 -----------------
uniq_genes <- function(x) sort(unique(unlist(strsplit(paste(x, collapse=";"), ";"))))
conn <- detail %>%
  group_by(celltype, direction, database, pathway, subclass) %>%
  summarise(anchored_stages = list(unique(stage)),
            sig_anchored_stages = list(unique(stage[is_significant])),
            n_anchor_union = length(uniq_genes(anchor_genes)),
            anchor_genes = paste(uniq_genes(anchor_genes), collapse=";"),
            .groups="drop") %>%
  left_join(tested_tbl, by=c("celltype","database","pathway"))
cat("连接聚合行数（未过滤）:", nrow(conn), "\n")

In [ ]:
# ---- 规则区（连续锚定 + U1 显著性，日后改档只动这里）------------------------
# DM/DAL: dev-onset -> Young 连续; AAM: Young & Old 锚定
eval_conn <- function(celltype, subclass, anchored, sig_anchored, tested) {
  stg <- avail[[celltype]]
  ai <- which(stg %in% anchored); ti <- which(stg %in% tested)
  yi <- match("Young", stg); oi <- match("Old", stg)
  cont <- function(from, to) { sp <- intersect(from:to, ti); length(sp)>0 && all(sp %in% ai) }
  sig_dev   <- any(sig_anchored %in% dev_stages)
  sig_adult <- any(sig_anchored %in% adult_stages)
  if (subclass == "AAM") {
    pass_anchor <- (yi %in% ai) && (oi %in% ai)
    return(list(pass_anchor=pass_anchor, onset=NA_character_,
                pass_U1 = pass_anchor && sig_adult))
  }
  di <- ai[stg[ai] %in% dev_stages]                 # 发育期锚定
  if (length(di) == 0) return(list(pass_anchor=FALSE, onset=NA_character_, pass_U1=FALSE))
  on <- min(di)
  pass_anchor <- cont(on, yi)                       # onset -> Young 连续
  list(pass_anchor=pass_anchor, onset=stg[on],
       pass_U1 = pass_anchor && sig_dev && sig_adult)
}

ev <- pmap(list(conn$celltype, conn$subclass, conn$anchored_stages,
                conn$sig_anchored_stages, conn$tested_stages), eval_conn)
conn <- conn %>% mutate(pass_anchor = map_lgl(ev, "pass_anchor"),
                        onset_stage = map_chr(ev, "onset"),
                        pass_U1     = map_lgl(ev, "pass_U1"))

In [ ]:
# ---- 附标注列：五期 NES + retention + context + 显著性明细 -------------------
conn <- conn %>%
  mutate(sig_dev   = map_lgl(sig_anchored_stages, ~any(.x %in% dev_stages)),
         sig_adult = map_lgl(sig_anchored_stages, ~any(.x %in% adult_stages)),
         n_sig_anchored = lengths(sig_anchored_stages),
         stages_anchored = map_chr(anchored_stages, ~paste(.x, collapse=";")),
         sig_anchored = map_chr(sig_anchored_stages, ~paste(.x, collapse=";"))) %>%
  left_join(nes_wide, by=c("celltype","database","pathway")) %>%
  mutate(NES_retention = ifelse(!is.na(NES_Young) & NES_Young>0, round(NES_Old/NES_Young,3), NA),
         context_class = case_when(is.na(NES_retention) ~ NA_character_,
                                   NES_retention >= 0.5 ~ "coherent", TRUE ~ "divergent")) %>%
  select(-anchored_stages, -sig_anchored_stages, -tested_stages)

write_csv(conn, file.path(out_dir, "2026-09-02_2_fate_pathway_connection_all.csv"), na="NA")
conn_u1 <- conn %>% filter(pass_U1)
cat("pass_anchor:", sum(conn$pass_anchor), "| pass_U1:", nrow(conn_u1), "\n")

In [ ]:
# ---- 特有/共享分解（U1 集合）------------------------------------------------
combo <- conn_u1 %>%
  group_by(celltype, direction, database, pathway) %>%
  summarise(subclass_combo = paste(sort(unique(subclass)), collapse="+"),
            n_subclass = n_distinct(subclass), .groups="drop")
write_csv(combo, file.path(out_dir, "2026-09-02_2_pathway_fate_combo.csv"), na="NA")
combo_counts <- combo %>% count(direction, subclass_combo, celltype, name="n_pathways")
write_csv(combo_counts, file.path(out_dir, "2026-09-02_2_fate_combo_counts.csv"), na="NA")

# ---- 删除原因诊断（methods 用；仅统计通过连续锚定但被 U1 显著性刷掉的）------
del <- conn %>% filter(pass_anchor, !pass_U1) %>%
  mutate(reason = case_when(subclass=="AAM" ~ "AAM_no_adult_sig",
                            !sig_dev & !sig_adult ~ "no_sig_any",
                            !sig_dev ~ "no_dev_sig", TRUE ~ "no_adult_sig")) %>%
  count(direction, subclass, reason, name="n")
write_csv(del, file.path(out_dir, "2026-09-02_2_U1_deletion_diagnostics.csv"), na="NA")

In [ ]:
# ---- 汇总打印 --------------------------------------------------------------
cat("=== U1 连接数（Macular）===\n")
conn_u1 %>% filter(direction=="Macular direction") %>% count(subclass, celltype) %>%
  pivot_wider(names_from=celltype, values_from=n, values_fill=0) %>% as.data.frame() %>% print()

cat("\n=== 特有/共享分解（Macular，U1 pathway 数）===\n")
combo_counts %>% filter(direction=="Macular direction") %>%
  pivot_wider(names_from=celltype, values_from=n_pathways, values_fill=0) %>% as.data.frame() %>% print()

cat("\n=== 唯一 pathway 数（module 工作量）===\n")
cat("Macular:", n_distinct(conn_u1$pathway[conn_u1$direction=="Macular direction"]),
    "| Peripheral:", n_distinct(conn_u1$pathway[conn_u1$direction=="Peripheral direction"]),
    "| 并集:", n_distinct(conn_u1$pathway), "\n")

cat("\n=== U1 删除诊断（Macular）===\n")
del %>% filter(direction=="Macular direction") %>% as.data.frame() %>% print()

In [ ]:
# ---- 参数与 sessionInfo ----------------------------------------------------
writeLines(c("date_str: 2026-09-02_2  (线1 U1 连续锚定 + 时期匹配显著性)",
  "input: 2026-09-02_1_anchor_detail_long.csv + GSEA all_results (tested stages)",
  "anchoring window (continuous, union genes, direction auto by LE):",
  "  DM & DAL : dev-onset -> Young (pathway 层规则相同; 命运由锚定基因决定)",
  "  AAM      : Young AND Old anchored",
  "  not_testable 不打断; tested-but-unanchored 打断",
  "U1 significance (padj<0.05, direction-consistent, within anchored window):",
  "  DM/DAL (dev-onset): >=1 dev sig AND >=1 adult(Young/Old) sig",
  "  AAM   (adult-onset): >=1 adult sig",
  "wording: enrichment ok (has significance support); interpretation unit = module",
  "annotations kept: 五期NES, NES_retention, context_class, sig_dev/sig_adult, anchor genes",
  "", capture.output(sessionInfo())),
  file.path(out_dir, "2026-09-02_2_parameters_sessionInfo.txt"))
cat("完成。输出目录:", out_dir, "\n")